# RAG Evaluation Analysis Tutorial

This notebook provides interactive analysis and visualization of RAG evaluation results.

## Features
- Load and explore evaluation results
- Compare performance across RAG methods (Naive, KG, Hybrid)
- Generate visualizations and statistical insights
- Export results to CSV

## Setup

First, make sure you have evaluation results in `benchmarks/results/`

In [ ]:
# Import required libraries
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from glob import glob

# Add src to path
sys.path.insert(0, str(Path.cwd().parent))

# Import analysis modules
from src.analysis.metrics import (
    load_evaluation_results,
    extract_detailed_results_to_dataframe,
    calculate_summary_stats,
    compare_methods,
    generate_text_report,
    calculate_per_question_metrics,
    calculate_score_differences,
)
from src.analysis.visualizations import (
    plot_score_comparison,
    plot_latency_comparison,
    plot_chunk_count_distribution,
    plot_comprehensive_comparison,
    plot_per_question_heatmap,
)

# Configure plotting
%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("✓ Setup complete!")

ValidationError: 1 validation error for BenchmarkSettings
sample_seed
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='42  # For reproducibility', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/int_parsing

## 1. Load Evaluation Results

Find and load evaluation result files from the benchmarks directory.

In [ ]:
# Find all evaluation result files
results_dir = Path('../benchmarks/results')
result_files = sorted(results_dir.glob('eval_*.json'))

print(f"Found {len(result_files)} evaluation result file(s):\n")
for i, file in enumerate(result_files, 1):
    print(f"{i}. {file.name}")

# Select which file to analyze (change index to select different file)
selected_file = result_files[-1] if result_files else None

if selected_file:
    print(f"\n✓ Selected: {selected_file.name}")
else:
    print("\n⚠ No result files found! Run 'python cli.py evaluate' first.")

In [ ]:
# Load the selected results
if selected_file:
    results = load_evaluation_results(selected_file)
    
    print("="*80)
    print("EVALUATION METADATA")
    print("="*80)
    print(f"Evaluation ID: {results['evaluation_id']}")
    print(f"Dataset: {results['dataset_name']}")
    print(f"Timestamp: {results['timestamp']}")
    print(f"Number of Questions: {results['num_questions']}")
    print("\n✓ Results loaded successfully!")
else:
    print("⚠ No file selected")

## 2. Quick Summary

Get a high-level overview of method performance.

In [ ]:
# Generate comparison table
if selected_file:
    comparison_df = compare_methods(results)
    
    print("\n")
    print("="*80)
    print("METHOD COMPARISON")
    print("="*80)
    print(comparison_df.to_string(index=False))
    
    # Display as styled dataframe
    comparison_df.style.background_gradient(subset=['Overall Score'], cmap='RdYlGn', vmin=0, vmax=5)

## 3. Detailed Results DataFrame

Extract all detailed results into a pandas DataFrame for analysis.

In [ ]:
# Extract detailed results
if selected_file:
    df = extract_detailed_results_to_dataframe(results)
    
    print(f"Loaded {len(df)} detailed results\n")
    print("DataFrame columns:")
    print(df.columns.tolist())
    
    # Display first few rows
    print("\nFirst 5 rows:")
    df.head()

In [ ]:
# Summary statistics
if selected_file:
    stats_df = calculate_summary_stats(df)
    
    print("="*80)
    print("SUMMARY STATISTICS")
    print("="*80)
    print(stats_df.to_string(index=False))
    
    stats_df

## 4. Score Visualizations

Compare quality scores across methods.

In [ ]:
# Plot score comparison
if selected_file:
    fig = plot_score_comparison(df)
    plt.show()

## 5. Latency Analysis

Compare response times across methods.

In [ ]:
# Plot latency comparison
if selected_file:
    fig = plot_latency_comparison(df)
    plt.show()

## 6. Retrieved Chunks Analysis

Analyze how many chunks each method retrieves.

In [ ]:
# Plot chunk distribution
if selected_file:
    fig = plot_chunk_count_distribution(df)
    plt.show()

## 7. Comprehensive Dashboard

View all metrics in a single comprehensive visualization.

In [ ]:
# Generate comprehensive dashboard
if selected_file:
    fig = plot_comprehensive_comparison(df)
    plt.show()

## 8. Per-Question Performance Heatmap

See how each method performs on individual questions.

In [ ]:
# Generate heatmap
if selected_file:
    fig = plot_per_question_heatmap(df)
    plt.show()

## 9. Per-Question Detailed Metrics

Analyze performance question by question.

In [ ]:
# Calculate per-question metrics
if selected_file:
    per_question_df = calculate_per_question_metrics(df)
    
    print("="*80)
    print("PER-QUESTION METRICS")
    print("="*80)
    print(per_question_df.to_string(index=False))
    
    per_question_df

## 10. Score Differences Between Methods

Calculate how much better/worse each method performs relative to others.

In [ ]:
# Calculate score differences
if selected_file:
    diff_df = calculate_score_differences(df)
    
    print("="*80)
    print("SCORE DIFFERENCES (Positive = Improvement)")
    print("="*80)
    print(diff_df.to_string(index=False))
    
    diff_df

## 11. Custom Analysis

Perform your own custom analysis using the DataFrame.

In [ ]:
# Example: Plot average scores by method
if selected_file:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    avg_by_method = df.groupby('method')['avg_score'].mean().sort_values(ascending=False)
    
    bars = ax.bar(avg_by_method.index, avg_by_method.values, 
                   color=['#3498db', '#e74c3c', '#2ecc71'])
    
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
               f'{height:.2f}',
               ha='center', va='bottom', fontweight='bold', fontsize=12)
    
    ax.set_ylabel('Average Score', fontsize=12, fontweight='bold')
    ax.set_title('Average Overall Score by Method', fontsize=14, fontweight='bold', pad=20)
    ax.set_ylim(0, 5.5)
    ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Example: Correlation between latency and score
if selected_file:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    colors = {'Naive Rag': '#3498db', 'Kg Rag': '#e74c3c', 'Hybrid Rag': '#2ecc71'}
    
    for method in df['method'].unique():
        method_data = df[df['method'] == method]
        ax.scatter(method_data['latency_ms'], method_data['avg_score'],
                  label=method, alpha=0.7, s=150, color=colors.get(method, 'gray'))
    
    ax.set_xlabel('Latency (ms)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Average Score', fontsize=12, fontweight='bold')
    ax.set_title('Latency vs Quality Tradeoff', fontsize=14, fontweight='bold', pad=20)
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Calculate correlation
    correlation = df['latency_ms'].corr(df['avg_score'])
    print(f"\nCorrelation between latency and score: {correlation:.3f}")
    print("(Negative = higher latency tends to give better scores)")

## 12. Export Results

Save your analysis results to CSV files.

In [ ]:
# Export to CSV
if selected_file:
    output_dir = Path('../benchmarks/results/notebook_exports')
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Export detailed results
    df.to_csv(output_dir / 'detailed_results.csv', index=False)
    print(f"✓ Saved: {output_dir / 'detailed_results.csv'}")
    
    # Export summary statistics
    stats_df = calculate_summary_stats(df)
    stats_df.to_csv(output_dir / 'summary_statistics.csv', index=False)
    print(f"✓ Saved: {output_dir / 'summary_statistics.csv'}")
    
    # Export comparison table
    comparison_df = compare_methods(results)
    comparison_df.to_csv(output_dir / 'method_comparison.csv', index=False)
    print(f"✓ Saved: {output_dir / 'method_comparison.csv'}")
    
    # Export per-question metrics
    per_question_df = calculate_per_question_metrics(df)
    per_question_df.to_csv(output_dir / 'per_question_metrics.csv', index=False)
    print(f"✓ Saved: {output_dir / 'per_question_metrics.csv'}")
    
    print(f"\n✓ All CSVs exported to: {output_dir}")

## 13. Generate Text Report

Create a comprehensive text summary of the evaluation.

In [ ]:
# Generate and display text report
if selected_file:
    report = generate_text_report(results)
    print(report)
    
    # Save to file
    output_dir = Path('../benchmarks/results/notebook_exports')
    output_dir.mkdir(parents=True, exist_ok=True)
    
    report_path = output_dir / 'analysis_report.txt'
    report_path.write_text(report)
    print(f"\n✓ Report saved to: {report_path}")

## Conclusion

You've completed the RAG evaluation analysis! 

### Key Takeaways

1. **Method Comparison**: Compare overall performance across Naive, KG, and Hybrid RAG
2. **Quality Metrics**: Analyze correctness, completeness, relevance, faithfulness, and clarity
3. **Performance**: Evaluate latency and efficiency tradeoffs
4. **Retrieval**: Understand chunk retrieval patterns

### Next Steps

- Run evaluations on different datasets
- Experiment with different chunk sizes or retrieval parameters
- Compare results across multiple evaluation runs
- Create custom visualizations for specific metrics

### Resources

- Documentation: `docs/EVALUATION_PIPELINE_EXPLAINED.md`
- Analysis code: `src/analysis/`
- CLI tool: `python cli.py analyze --help`